In [1]:
import pandas as pd
import numpy as np
import joblib
import requests
import holidays

In [2]:
MODEL_PATH = "spot_price_forecast_model.pkl"

SPOT_HISTORY_PATH = "SPOT_2026.xlsx"

FORECAST_RUN_TIME = pd.Timestamp("2026-03-28 08:00:00")
FORECAST_DATE = pd.Timestamp("2026-03-29").date()

LAT = 52.2297
LON = 21.0122

In [3]:
model_package = joblib.load(MODEL_PATH)

model = model_package["best_model"]
features = model_package["features"]

print("Model:", model_package["best_model_name"])
print("Liczba cech:", len(features))

Model: XGBoost
Liczba cech: 60


In [4]:
spot_hist = pd.read_excel(SPOT_HISTORY_PATH)

spot_hist["timestamp"] = pd.to_datetime(spot_hist["timestamp"])
spot_hist["price_spot"] = pd.to_numeric(spot_hist["price_spot"], errors="coerce")

spot_hist = (
    spot_hist
    .dropna(subset=["timestamp", "price_spot"])
    .drop_duplicates(subset=["timestamp"])
    .sort_values("timestamp")
    .reset_index(drop=True)
)

# W dniu 2 lutego znamy już ceny na cały 2 lutego
last_known_price_time = pd.Timestamp(f"{FORECAST_DATE - pd.Timedelta(days=1)} 23:00:00")

spot_hist = spot_hist[spot_hist["timestamp"] <= last_known_price_time].copy()

spot_hist

,timestamp,price_spot
0,2026-03-08 00:00:00,550.00
1,2026-03-08 01:00:00,524.96
2,2026-03-08 02:00:00,508.13
3,2026-03-08 03:00:00,509.98
4,2026-03-08 04:00:00,538.00
...,...,...
499,2026-03-28 19:00:00,656.34
500,2026-03-28 20:00:00,598.17
501,2026-03-28 21:00:00,555.00
502,2026-03-28 22:00:00,510.03


In [5]:
import requests
import time

def get_pse_json(endpoint, params=None):
    url = f"https://api.raporty.pse.pl/api/{endpoint}"
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    data = response.json()
    
    if "value" in data:
        return pd.DataFrame(data["value"])
    
    return pd.DataFrame(data)

In [6]:
def download_pse_pk5l_wp(start_date, end_date):
    url = "https://api.raporty.pse.pl/api/pk5l-wp"
    
    start_date = pd.to_datetime(start_date).strftime("%Y-%m-%d")
    end_date_plus_1 = (pd.to_datetime(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    params = {
        "$filter": (
            f"plan_dtime ge '{start_date}' and "
            f"plan_dtime le '{end_date_plus_1}'"
        ),
        "$first": 10000
    }
    response = requests.get(url, params=params)

    print(response.status_code)
    print(response.url)

    response.raise_for_status()

    data = response.json()

    if "value" in data:
        return pd.DataFrame(data["value"])

    return pd.DataFrame(data)

In [7]:
forecast_start = pd.Timestamp(f"{FORECAST_DATE} 00:00:00")
forecast_end = pd.Timestamp(f"{FORECAST_DATE} 23:00:00")

history_start = forecast_start - pd.Timedelta(hours=240)

full_index = pd.date_range(
    start=history_start,
    end=forecast_end,
    freq="h"
)

base = pd.DataFrame({"timestamp": full_index})
base.head(), base.tail()

(            timestamp
 0 2026-03-19 00:00:00
 1 2026-03-19 01:00:00
 2 2026-03-19 02:00:00
 3 2026-03-19 03:00:00
 4 2026-03-19 04:00:00,
               timestamp
 259 2026-03-29 19:00:00
 260 2026-03-29 20:00:00
 261 2026-03-29 21:00:00
 262 2026-03-29 22:00:00
 263 2026-03-29 23:00:00)

In [8]:
pse_start = (forecast_start - pd.Timedelta(hours=240)).strftime("%Y-%m-%d")
pse_end = forecast_end.strftime("%Y-%m-%d")

In [9]:
load_raw = download_pse_pk5l_wp(
    start_date=pse_start,
    end_date=pse_end
)
load_raw.head(), load_raw.tail()

200
https://api.raporty.pse.pl/api/pk5l-wp?%24filter=plan_dtime+ge+%272026-03-19%27+and+plan_dtime+le+%272026-03-30%27&%24first=10000


(    period           plan_dtime  req_pow_res business_date  \
 0  23 - 24  2026-03-19 00:00:00       1579.0    2026-03-18   
 1  00 - 01  2026-03-19 01:00:00       1487.0    2026-03-19   
 2  01 - 02  2026-03-19 02:00:00       1446.0    2026-03-19   
 3  02 - 03  2026-03-19 03:00:00       1422.0    2026-03-19   
 4  03 - 04  2026-03-19 04:00:00       1428.0    2026-03-19   
 
         plan_dtime_utc           publication_ts  fcst_pv_tot_gen  \
 0  2026-03-18 23:00:00  2026-03-18 23:53:40.604              0.0   
 1  2026-03-19 00:00:00  2026-03-19 01:34:30.830              0.0   
 2  2026-03-19 01:00:00  2026-03-19 02:34:15.957              0.0   
 3  2026-03-19 02:00:00  2026-03-19 03:33:46.403              0.0   
 4  2026-03-19 03:00:00  2026-03-19 04:34:15.947              0.0   
 
    fcst_wi_tot_gen  fcst_unav_energy  grid_demand_fcst  ...  \
 0           1315.0              86.0           16822.0  ...   
 1            877.0              86.0           16040.0  ...   
 2          

In [10]:
pse_forecast = load_raw.copy()

# Dostosuj nazwy po sprawdzeniu load_raw.columns
pse_forecast = pse_forecast.rename(columns={
    "plan_dtime": "timestamp",
    "grid_demand_fcst": "load",
    "fcst_pv_tot_gen": "pv_gen",
    "fcst_wi_tot_gen": "fw_gen"
})

pse_forecast["timestamp_raw"] = pse_forecast["timestamp"].astype(str)

pse_forecast["timestamp"] = (
    pse_forecast["timestamp_raw"]
    .str.replace("03a", "03", regex=False)
)

pse_forecast["timestamp"] = pd.to_datetime(
    pse_forecast["timestamp"],
    errors="coerce"
)

for col in ["load", "pv_gen", "fw_gen"]:
    pse_forecast[col] = pd.to_numeric(pse_forecast[col], errors="coerce")

pse_forecast = (
    pse_forecast
    .dropna(subset=["timestamp", "load", "pv_gen", "fw_gen"])
    .groupby("timestamp", as_index=False)[["load", "pv_gen", "fw_gen"]]
    .mean()
    .sort_values("timestamp")
    .reset_index(drop=True)
)
pse_forecast = pse_forecast[["timestamp", "load", "pv_gen", "fw_gen"]]

In [11]:
pse_forecast.describe()

,timestamp,load,pv_gen,fw_gen
count,263,263.000000,263.000000,263.000000
mean,2026-03-24 11:04:47.452471296,16262.988593,2739.524715,1864.026616
min,2026-03-19 00:00:00,9285.000000,0.000000,41.000000
25%,2026-03-21 17:30:00,14527.000000,0.000000,643.500000
50%,2026-03-24 11:00:00,15978.000000,388.000000,1707.000000
75%,2026-03-27 04:30:00,17985.000000,5879.000000,2522.500000
max,2026-03-29 23:00:00,21846.000000,11431.000000,7180.000000
std,NaN,2647.760947,3457.394915,1502.650268


In [12]:
base = base.merge(
    spot_hist[["timestamp", "price_spot"]],
    on="timestamp",
    how="left"
)

base = base.merge(
    pse_forecast[["timestamp", "load", "pv_gen", "fw_gen"]],
    on="timestamp",
    how="left"
)

base.head()

,timestamp,price_spot,load,pv_gen,fw_gen
0,2026-03-19 00:00:00,484.00,16822.0,0.0,1315.0
1,2026-03-19 01:00:00,475.00,16040.0,0.0,877.0
2,2026-03-19 02:00:00,470.26,15595.0,0.0,542.0
3,2026-03-19 03:00:00,480.00,15469.0,0.0,307.0
4,2026-03-19 04:00:00,502.38,15683.0,0.0,241.0


In [13]:
cols_to_interpolate = ["load", "pv_gen", "fw_gen"]

base = base.sort_values("timestamp").copy()

base[cols_to_interpolate] = (
    base[cols_to_interpolate]
    .interpolate(method="linear", limit_direction="both")
)

print(base[cols_to_interpolate].isna().sum())

load      0
pv_gen    0
fw_gen    0
dtype: int64


In [14]:
print(base.isna().sum())

timestamp      0
price_spot    24
load           0
pv_gen         0
fw_gen         0
dtype: int64


In [15]:
def download_open_meteo_weather(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "wind_speed_10m",
            "cloud_cover"
        ],
        "timezone": "Europe/Warsaw"
    }
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    data = response.json()["hourly"]
    weather = pd.DataFrame(data)
    weather["timestamp"] = pd.to_datetime(weather["time"])
    weather = weather.drop(columns=["time"])
    
    return weather

In [16]:
weather = download_open_meteo_weather(
    LAT,
    LON,
    base["timestamp"].min().strftime("%Y-%m-%d"),
    base["timestamp"].max().strftime("%Y-%m-%d")
)

base = base.merge(weather, on="timestamp", how="left")

base.head()

,timestamp,price_spot,load,pv_gen,fw_gen,temperature_2m,relative_humidity_2m,wind_speed_10m,cloud_cover
0,2026-03-19 00:00:00,484.00,16822.0,0.0,1315.0,6.2,75,8.5,0
1,2026-03-19 01:00:00,475.00,16040.0,0.0,877.0,5.2,79,8.2,0
2,2026-03-19 02:00:00,470.26,15595.0,0.0,542.0,4.7,80,7.9,10
3,2026-03-19 03:00:00,480.00,15469.0,0.0,307.0,3.8,83,7.5,20
4,2026-03-19 04:00:00,502.38,15683.0,0.0,241.0,3.1,86,7.0,27


In [17]:
df_pred = base.copy()

df_pred["year"] = df_pred["timestamp"].dt.year
df_pred["month"] = df_pred["timestamp"].dt.month
df_pred["day"] = df_pred["timestamp"].dt.day
df_pred["hour"] = df_pred["timestamp"].dt.hour
df_pred["dayofweek"] = df_pred["timestamp"].dt.dayofweek
df_pred["dayofyear"] = df_pred["timestamp"].dt.dayofyear
df_pred["is_weekend"] = df_pred["dayofweek"].isin([5, 6]).astype(int)

df_pred["hour_sin"] = np.sin(2 * np.pi * df_pred["hour"] / 24)
df_pred["hour_cos"] = np.cos(2 * np.pi * df_pred["hour"] / 24)

df_pred["month_sin"] = np.sin(2 * np.pi * df_pred["month"] / 12)
df_pred["month_cos"] = np.cos(2 * np.pi * df_pred["month"] / 12)

df_pred["dayofyear_sin"] = np.sin(2 * np.pi * df_pred["dayofyear"] / 365)
df_pred["dayofyear_cos"] = np.cos(2 * np.pi * df_pred["dayofyear"] / 365)

In [18]:
df_pred["res_gen"] = df_pred["fw_gen"] + df_pred["pv_gen"]
df_pred["residual_load"] = df_pred["load"] - df_pred["res_gen"]

df_pred["res_share"] = df_pred["res_gen"] / df_pred["load"]
df_pred["pv_share"] = df_pred["pv_gen"] / df_pred["load"]
df_pred["fw_share"] = df_pred["fw_gen"] / df_pred["load"]

In [19]:
BASE_TEMP_HEATING = 18
BASE_TEMP_COOLING = 22

df_pred["heating_degree"] = np.maximum(0, BASE_TEMP_HEATING - df_pred["temperature_2m"])
df_pred["cooling_degree"] = np.maximum(0, df_pred["temperature_2m"] - BASE_TEMP_COOLING)

In [20]:
pl_holidays = holidays.Poland(years=sorted(df_pred["year"].unique()))

df_pred["date"] = df_pred["timestamp"].dt.date
df_pred["is_holiday"] = df_pred["date"].isin(pl_holidays).astype(int)

df_pred["is_weekend_or_holiday"] = (
    (df_pred["is_weekend"] == 1) |
    (df_pred["is_holiday"] == 1)
).astype(int)

In [21]:
price_lags = [24, 48, 72, 168]

for lag in price_lags:
    df_pred[f"price_lag_{lag}h"] = df_pred["price_spot"].shift(lag)

In [22]:
system_lags = [24, 48, 72, 168]

for lag in system_lags:
    df_pred[f"load_lag_{lag}h"] = df_pred["load"].shift(lag)
    df_pred[f"pv_lag_{lag}h"] = df_pred["pv_gen"].shift(lag)
    df_pred[f"fw_lag_{lag}h"] = df_pred["fw_gen"].shift(lag)
    df_pred[f"residual_load_lag_{lag}h"] = df_pred["residual_load"].shift(lag)

In [23]:
rolling_windows = [24, 48, 72, 168]

for window in rolling_windows:
    df_pred[f"price_roll_mean_{window}h"] = (
        df_pred["price_spot"].shift(24).rolling(window).mean()
    )
    
    df_pred[f"price_roll_std_{window}h"] = (
        df_pred["price_spot"].shift(24).rolling(window).std()
    )
    
    df_pred[f"residual_load_roll_mean_{window}h"] = (
        df_pred["residual_load"].shift(24).rolling(window).mean()
    )
    
    df_pred[f"load_roll_mean_{window}h"] = (
        df_pred["load"].shift(24).rolling(window).mean()
    )

In [24]:
forecast_rows = df_pred[
    (df_pred["timestamp"] >= forecast_start) &
    (df_pred["timestamp"] <= forecast_end)
].copy()

missing_features = [col for col in features if col not in forecast_rows.columns]

if missing_features:
    raise ValueError(f"Brakuje cech wymaganych przez model: {missing_features}")

X_forecast = forecast_rows[features].copy()

print("Braki w X_forecast:")
print(X_forecast.isna().sum()[X_forecast.isna().sum() > 0])

Braki w X_forecast:
Series([], dtype: int64)


In [25]:
forecast_rows["price_spot_forecast"] = model.predict(X_forecast)

spot_forecast_d1 = forecast_rows[[
    "timestamp",
    "price_spot_forecast",
    "load",
    "pv_gen",
    "fw_gen",
    "residual_load",
    "temperature_2m"
]].copy()

spot_forecast_d1

,timestamp,price_spot_forecast,load,pv_gen,fw_gen,residual_load,temperature_2m
240,2026-03-29 00:00:00,478.456757,14982.0,0.0,1778.0,13204.0,9.1
241,2026-03-29 01:00:00,431.110413,13850.0,0.0,1966.0,11884.0,8.4
242,2026-03-29 02:00:00,417.909973,13534.0,0.0,2089.0,11445.0,7.8
243,2026-03-29 03:00:00,400.419159,13218.0,0.0,2212.0,11006.0,7.1
244,2026-03-29 04:00:00,362.720062,12737.0,0.0,2416.0,10321.0,6.5
245,2026-03-29 05:00:00,363.948761,12807.0,0.0,2538.0,10269.0,5.8
246,2026-03-29 06:00:00,374.500122,12975.0,0.0,2515.0,10460.0,5.2
247,2026-03-29 07:00:00,387.726105,13429.0,21.0,2537.0,10871.0,4.8
248,2026-03-29 08:00:00,365.454437,13654.0,616.0,2398.0,10640.0,5.7
249,2026-03-29 09:00:00,307.317169,13565.0,2203.0,1946.0,9416.0,7.2


In [26]:
spot_forecast_d1.to_excel(
    f"prognoza_SPOT_{FORECAST_DATE}.xlsx",
    index=False
)
print(f"Zapisano prognozę dla dnia {FORECAST_DATE}")

Zapisano prognozę dla dnia 2026-03-29
